## Install Dependencies


In [1]:
!pip install ultralytics opencv-python matplotlib pyyaml


## Import Libraries


In [2]:
import os
import cv2
import yaml
import random
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

## Data Loading

In [3]:
import os
import cv2
import numpy as np
from pathlib import Path
import pandas as pd
from tqdm import tqdm

DATASET_ROOT = Path("../dataset")
SPLITS = ["train", "valid", "test"]

def verify_image_label_pairs(split):
    """
    Checks that every image has a matching label file and vice versa.
    Reports any mismatches.
    """
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    image_stems = set(p.stem for p in img_dir.glob("*") if p.suffix in [".jpg", ".jpeg", ".png"])
    label_stems = set(p.stem for p in lbl_dir.glob("*.txt"))

    images_missing_labels = image_stems - label_stems
    labels_missing_images = label_stems - image_stems

    print(f"\n--- [{split.upper()}] ---")
    print(f"  Images found     : {len(image_stems)}")
    print(f"  Labels found     : {len(label_stems)}")

    if images_missing_labels:
        print(f"  WARNING: {len(images_missing_labels)} images have NO label file:")
        for name in list(images_missing_labels)[:5]:
            print(f"    - {name}")
    else:
        print("  All images have matching labels.")

    if labels_missing_images:
        print(f"  WARNING: {len(labels_missing_images)} labels have NO image file:")
        for name in list(labels_missing_images)[:5]:
            print(f"    - {name}")
    else:
        print("  All labels have matching images.")

    return image_stems, label_stems


def load_dataset_summary():
    """
    Loads all images and labels, builds a summary DataFrame with
    image dimensions, number of annotations per image, etc.
    """
    records = []

    for split in SPLITS:
        img_dir = DATASET_ROOT / split / "images"
        lbl_dir = DATASET_ROOT / split / "labels"

        image_files = list(img_dir.glob("*"))
        image_files = [f for f in image_files if f.suffix in [".jpg", ".jpeg", ".png"]]

        for img_path in tqdm(image_files, desc=f"Loading {split}"):
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"  CORRUPT IMAGE: {img_path.name} — could not be read")
                continue

            h, w = img.shape[:2]
            lbl_path = lbl_dir / (img_path.stem + ".txt")

            num_annotations = 0
            boxes = []
            if lbl_path.exists():
                with open(lbl_path) as f:
                    lines = [l.strip() for l in f.readlines() if l.strip()]
                    num_annotations = len(lines)
                    for line in lines:
                        parts = line.split()
                        if len(parts) == 5:
                            cls, cx, cy, bw, bh = map(float, parts)
                            boxes.append((cls, cx, cy, bw, bh))

            records.append({
                "split": split,
                "filename": img_path.name,
                "width": w,
                "height": h,
                "aspect_ratio": round(w / h, 2),
                "num_annotations": num_annotations,
                "has_pothole": num_annotations > 0,
            })

    df = pd.DataFrame(records)
    return df


def check_label_format(split):
    """
    Validates YOLO label format: each line must have 5 values,
    all normalized between 0 and 1, class index must be integer.
    """
    lbl_dir = DATASET_ROOT / split / "labels"
    errors = []

    for lbl_path in lbl_dir.glob("*.txt"):
        with open(lbl_path) as f:
            for i, line in enumerate(f.readlines()):
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    errors.append(f"{lbl_path.name} line {i+1}: expected 5 values, got {len(parts)}")
                    continue
                try:
                    cls_id = int(parts[0])
                    vals = list(map(float, parts[1:]))
                    for v in vals:
                        if not (0.0 <= v <= 1.0):
                            errors.append(f"{lbl_path.name} line {i+1}: value {v} out of [0,1] range")
                except ValueError:
                    errors.append(f"{lbl_path.name} line {i+1}: non-numeric value")

    if errors:
        print(f"\n  LABEL FORMAT ERRORS in {split}:")
        for e in errors[:10]:
            print(f"    {e}")
    else:
        print(f"  [{split}] All label files are correctly formatted.")

    return errors


if __name__ == "__main__":
    print("=" * 50)
    print("STEP 1: DATA LOADING & VERIFICATION")
    print("=" * 50)

    # 1. Check image-label pairing
    for split in SPLITS:
        verify_image_label_pairs(split)

    # 2. Check label format validity
    print("\n--- Label Format Check ---")
    for split in SPLITS:
        check_label_format(split)

    # 3. Build summary dataframe
    print("\n--- Building Dataset Summary ---")
    df = load_dataset_summary()
    df.to_csv("../outputs/preprocessed/dataset_summary.csv", index=False)

    print("\n--- Dataset Summary ---")
    print(df.groupby("split").agg(
        total_images=("filename", "count"),
        images_with_pothole=("has_pothole", "sum"),
        total_annotations=("num_annotations", "sum"),
        avg_annotations=("num_annotations", "mean"),
    ).round(2))

    print("\nStep 1 complete. Summary saved to outputs/preprocessed/dataset_summary.csv")

STEP 1: DATA LOADING & VERIFICATION

--- [TRAIN] ---
  Images found     : 1906
  Labels found     : 1906
  All images have matching labels.
  All labels have matching images.

--- [VALID] ---
  Images found     : 542
  Labels found     : 542
  All images have matching labels.
  All labels have matching images.

--- [TEST] ---
  Images found     : 274
  Labels found     : 274
  All images have matching labels.
  All labels have matching images.

--- Label Format Check ---
  [train] All label files are correctly formatted.
  [valid] All label files are correctly formatted.
  [test] All label files are correctly formatted.

--- Building Dataset Summary ---


Loading test: 100%|██████████| 274/274 [00:00<00:00, 295.35it/s]


--- Dataset Summary ---
       total_images  images_with_pothole  total_annotations  avg_annotations
split                                                                       
test            274                  274                861             3.14
train          1906                 1905               6564             3.44
valid           542                  542               1853             3.42

Step 1 complete. Summary saved to outputs/preprocessed/dataset_summary.csv


## Data Preprocessing

In [4]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

DATASET_ROOT = Path("../dataset")
CLEAN_ROOT   = Path("../dataset_clean")
SPLITS       = ["train", "valid", "test"]

issues_log = []


def is_image_valid(img_path):
    img = cv2.imread(str(img_path))
    return img is not None


def remove_duplicate_labels(lbl_path):
    with open(lbl_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    unique_lines = list(dict.fromkeys(lines))
    if len(unique_lines) < len(lines):
        issues_log.append(f"Duplicates removed: {lbl_path.name} ({len(lines) - len(unique_lines)} removed)")
        with open(lbl_path, "w") as f:
            f.write("\n".join(unique_lines) + "\n")


def clamp_bbox_values(lbl_path):
    with open(lbl_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    fixed_lines = []
    changed = False
    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            fixed_lines.append(line)
            continue
        cls = parts[0]
        vals = [float(v) for v in parts[1:]]
        clamped = [max(0.001, min(0.999, v)) for v in vals]
        if clamped != vals:
            changed = True
        fixed_lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in clamped))
    if changed:
        issues_log.append(f"BBox clamped: {lbl_path.name}")
        with open(lbl_path, "w") as f:
            f.write("\n".join(fixed_lines) + "\n")


def check_image_size_uniformity(split):
    img_dir = DATASET_ROOT / split / "images"
    sizes = set()
    for img_path in img_dir.glob("*"):
        if img_path.suffix in [".jpg", ".jpeg", ".png"]:
            img = cv2.imread(str(img_path))
            if img is not None:
                sizes.add(img.shape[:2])
    if len(sizes) == 1:
        print(f"  [{split}] All images same size: {sizes.pop()}")
    else:
        print(f"  [{split}] Mixed sizes detected ({len(sizes)} unique sizes). YOLO will resize automatically.")


def copy_and_clean_dataset():
    if CLEAN_ROOT.exists():
        shutil.rmtree(CLEAN_ROOT)
    CLEAN_ROOT.mkdir(parents=True)

    for split in SPLITS:
        src_img_dir = DATASET_ROOT / split / "images"
        src_lbl_dir = DATASET_ROOT / split / "labels"
        dst_img_dir = CLEAN_ROOT / split / "images"
        dst_lbl_dir = CLEAN_ROOT / split / "labels"
        dst_img_dir.mkdir(parents=True)
        dst_lbl_dir.mkdir(parents=True)

        image_files = [f for f in src_img_dir.glob("*") if f.suffix in [".jpg", ".jpeg", ".png"]]
        kept, skipped = 0, 0

        for img_path in tqdm(image_files, desc=f"Cleaning {split}"):
            lbl_path = src_lbl_dir / (img_path.stem + ".txt")

            if not is_image_valid(img_path):
                issues_log.append(f"CORRUPT: {img_path.name}")
                skipped += 1
                continue

            dst_img = dst_img_dir / img_path.name
            shutil.copy2(img_path, dst_img)

            dst_lbl = dst_lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                shutil.copy2(lbl_path, dst_lbl)
                remove_duplicate_labels(dst_lbl)
                clamp_bbox_values(dst_lbl)
            else:
                dst_lbl.touch()

            kept += 1

        print(f"  [{split}] Kept: {kept}  |  Skipped (corrupt): {skipped}")

    log_path = Path("../outputs/preprocessed/cleaning_log.txt")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w") as f:
        f.write("\n".join(issues_log) if issues_log else "No issues found. Dataset is clean.")
    print(f"\nCleaning log saved to: {log_path}")


def update_yaml_for_clean_dataset():
    yaml_content = """path: ../dataset_clean
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['crocodile crack', 'longitudinal crack', 'pothole']
"""
    with open("../config/pothole.yaml", "w") as f:
        f.write(yaml_content)
    print("Updated config/pothole.yaml to point to dataset_clean/")


def load_dataset_summary():
    """
    Rebuilds dataset summary from the CLEAN dataset
    so that EDA reflects accurate, cleaned data.
    """
    records = []
    for split in SPLITS:
        img_dir = CLEAN_ROOT / split / "images"
        lbl_dir = CLEAN_ROOT / split / "labels"

        image_files = [f for f in img_dir.glob("*") if f.suffix in [".jpg", ".jpeg", ".png"]]

        for img_path in tqdm(image_files, desc=f"Building summary for {split}"):
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            h, w = img.shape[:2]
            lbl_path = lbl_dir / (img_path.stem + ".txt")

            num_annotations = 0
            if lbl_path.exists():
                with open(lbl_path) as f:
                    lines = [l.strip() for l in f if l.strip()]
                    num_annotations = len(lines)

            records.append({
                "split"          : split,
                "filename"       : img_path.name,
                "width"          : w,
                "height"         : h,
                "aspect_ratio"   : round(w / h, 2),
                "num_annotations": num_annotations,
                "has_pothole"    : num_annotations > 0,
            })

    return pd.DataFrame(records)


if __name__ == "__main__":
    print("=" * 50)
    print("STEP 2: PREPROCESSING & CLEANING")
    print("=" * 50)

    print("\n--- Checking image size uniformity ---")
    for split in SPLITS:
        check_image_size_uniformity(split)

    print("\n--- Copying and cleaning dataset ---")
    copy_and_clean_dataset()

    print("\n--- Updating YAML config ---")
    update_yaml_for_clean_dataset()

    print("\n--- Rebuilding dataset summary from clean data ---")
    df_clean = load_dataset_summary()
    out_path = Path("../outputs/preprocessed/dataset_summary.csv")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_csv(out_path, index=False)

    print("\n--- Clean Dataset Summary ---")
    print(df_clean.groupby("split").agg(
        total_images     = ("filename", "count"),
        images_with_defect = ("has_pothole", "sum"),
        total_annotations  = ("num_annotations", "sum"),
        avg_annotations    = ("num_annotations", "mean"),
    ).round(2))

    if issues_log:
        print(f"\n{len(issues_log)} issues found and fixed. Check outputs/preprocessed/cleaning_log.txt")
    else:
        print("\nDataset is clean. No issues found.")

    print("\nStep 2 complete.")

STEP 2: PREPROCESSING & CLEANING

--- Checking image size uniformity ---
  [train] All images same size: (640, 640)
  [valid] All images same size: (640, 640)
  [test] All images same size: (640, 640)

--- Copying and cleaning dataset ---


Cleaning train: 100%|██████████| 1906/1906 [00:43<00:00, 43.97it/s]


  [train] Kept: 1906  |  Skipped (corrupt): 0


Cleaning valid: 100%|██████████| 542/542 [00:12<00:00, 43.53it/s]


  [valid] Kept: 542  |  Skipped (corrupt): 0


Cleaning test: 100%|██████████| 274/274 [00:05<00:00, 46.29it/s]


  [test] Kept: 274  |  Skipped (corrupt): 0

Cleaning log saved to: ..\outputs\preprocessed\cleaning_log.txt

--- Updating YAML config ---
Updated config/pothole.yaml to point to dataset_clean/

--- Rebuilding dataset summary from clean data ---


Building summary for test: 100%|██████████| 274/274 [00:05<00:00, 47.99it/s]


--- Clean Dataset Summary ---
       total_images  images_with_defect  total_annotations  avg_annotations
split                                                                      
test            274                 274                861             3.14
train          1906                1905               6564             3.44
valid           542                 542               1853             3.42

1 issues found and fixed. Check outputs/preprocessed/cleaning_log.txt

Step 2 complete.


## EDA

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import numpy as np
import cv2
import random
from pathlib import Path

DATASET_ROOT = Path("../dataset_clean")       # EDA runs on CLEAN data
OUTPUT_DIR   = Path("../outputs/eda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES  = {0: "crocodile crack", 1: "longitudinal crack", 2: "pothole"}
COLORS       = {0: "red", 1: "blue", 2: "green"}
SPLITS       = ["train", "valid", "test"]

df = pd.read_csv("../outputs/preprocessed/dataset_summary.csv")


def plot_split_distribution():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    split_counts = df.groupby("split")["filename"].count()
    axes[0].bar(split_counts.index, split_counts.values, color=["#4A90E2", "#7ED321", "#F5A623"])
    axes[0].set_title("Total images per split")
    axes[0].set_ylabel("Count")
    for i, v in enumerate(split_counts.values):
        axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

    counts = df.groupby(["split", "has_pothole"])["filename"].count().unstack(fill_value=0)
    counts.columns = ["No defect", "Has defect"]
    counts.plot(kind="bar", ax=axes[1], color=["#E74C3C", "#2ECC71"], rot=0)
    axes[1].set_title("Defect vs background images per split")
    axes[1].set_ylabel("Count")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "01_split_distribution.png", dpi=150)
    plt.close()
    print("Saved: 01_split_distribution.png")


def plot_class_distribution():
    """Shows annotation count per class (crocodile crack, longitudinal crack, pothole) per split."""
    class_counts = {split: {0: 0, 1: 0, 2: 0} for split in SPLITS}

    for split in SPLITS:
        lbl_dir = DATASET_ROOT / split / "labels"
        for lbl_path in lbl_dir.glob("*.txt"):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        if cls_id in class_counts[split]:
                            class_counts[split][cls_id] += 1

    fig, ax = plt.subplots(figsize=(10, 5))
    x     = np.arange(len(CLASS_NAMES))
    width = 0.25

    for i, split in enumerate(SPLITS):
        counts = [class_counts[split][c] for c in range(3)]
        ax.bar(x + i * width, counts, width, label=split)

    ax.set_xticks(x + width)
    ax.set_xticklabels(CLASS_NAMES.values())
    ax.set_title("Annotation count per class per split")
    ax.set_ylabel("Number of annotations")
    ax.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "02_class_distribution.png", dpi=150)
    plt.close()
    print("Saved: 02_class_distribution.png")


def plot_annotation_stats():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, split in zip(axes, ["train", "valid"]):
        data = df[df["split"] == split]["num_annotations"]
        ax.hist(data, bins=range(0, int(data.max()) + 2), color="#4A90E2", edgecolor="white")
        ax.set_title(f"Annotations per image — {split}")
        ax.set_xlabel("Number of defects")
        ax.set_ylabel("Number of images")
        ax.axvline(data.mean(), color="red", linestyle="--", label=f"Mean: {data.mean():.1f}")
        ax.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "03_annotation_distribution.png", dpi=150)
    plt.close()
    print("Saved: 03_annotation_distribution.png")


def plot_image_dimensions():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    df["width"].hist(ax=axes[0], bins=20, color="#9B59B6", edgecolor="white")
    axes[0].set_title("Image width distribution")
    axes[0].set_xlabel("Width (px)")
    axes[0].set_ylabel("Count")

    df["height"].hist(ax=axes[1], bins=20, color="#E67E22", edgecolor="white")
    axes[1].set_title("Image height distribution")
    axes[1].set_xlabel("Height (px)")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "04_image_dimensions.png", dpi=150)
    plt.close()
    print("Saved: 04_image_dimensions.png")


def plot_bounding_box_stats():
    cx_list, cy_list, bw_list, bh_list = [], [], [], []

    for split in ["train", "valid"]:
        lbl_dir = DATASET_ROOT / split / "labels"
        for lbl_path in lbl_dir.glob("*.txt"):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = map(float, parts)
                        cx_list.append(cx)
                        cy_list.append(cy)
                        bw_list.append(bw)
                        bh_list.append(bh)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0][0].hist(bw_list, bins=30, color="#1ABC9C", edgecolor="white")
    axes[0][0].set_title("Bounding box width distribution (normalized)")
    axes[0][0].set_xlabel("Width")

    axes[0][1].hist(bh_list, bins=30, color="#3498DB", edgecolor="white")
    axes[0][1].set_title("Bounding box height distribution (normalized)")
    axes[0][1].set_xlabel("Height")

    axes[1][0].hist2d(cx_list, cy_list, bins=20, cmap="YlOrRd")
    axes[1][0].set_title("Bbox center heatmap (cx vs cy)")
    axes[1][0].set_xlabel("Center X")
    axes[1][0].set_ylabel("Center Y")

    areas = [bw * bh for bw, bh in zip(bw_list, bh_list)]
    axes[1][1].hist(areas, bins=30, color="#E74C3C", edgecolor="white")
    axes[1][1].set_title("Bounding box area distribution (normalized)")
    axes[1][1].set_xlabel("Area (w × h)")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "05_bbox_stats.png", dpi=150)
    plt.close()
    print("Saved: 05_bbox_stats.png")


def plot_sample_images(n=8):
    """Displays n random training images with color-coded bounding boxes per class."""
    img_dir = DATASET_ROOT / "train" / "images"
    lbl_dir = DATASET_ROOT / "train" / "labels"

    all_images = [f for f in img_dir.glob("*") if f.suffix in [".jpg", ".jpeg", ".png"]]
    samples    = random.sample(all_images, min(n, len(all_images)))

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes      = axes.flatten()

    for ax, img_path in zip(axes, samples):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        ax.imshow(img)

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id        = int(parts[0])
                        cx, cy, bw, bh = map(float, parts[1:])
                        x1 = (cx - bw / 2) * w
                        y1 = (cy - bh / 2) * h
                        color = COLORS.get(cls_id, "yellow")
                        rect  = patches.Rectangle(
                            (x1, y1), bw * w, bh * h,
                            linewidth=2, edgecolor=color, facecolor="none"
                        )
                        ax.add_patch(rect)
                        ax.text(x1, y1 - 4, CLASS_NAMES.get(cls_id, "unknown"),
                                color=color, fontsize=7, fontweight="bold")

        ax.set_title(img_path.name[:20], fontsize=8)
        ax.axis("off")

    # Legend
    legend_elements = [
        patches.Patch(edgecolor=COLORS[i], facecolor="none", label=CLASS_NAMES[i])
        for i in range(3)
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=10)
    plt.suptitle("Sample training images with ground truth bounding boxes", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "06_sample_images.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: 06_sample_images.png")


if __name__ == "__main__":
    print("=" * 50)
    print("STEP 3: EXPLORATORY DATA ANALYSIS (on clean data)")
    print("=" * 50)

    plot_split_distribution()
    plot_class_distribution()
    plot_annotation_stats()
    plot_image_dimensions()
    plot_bounding_box_stats()
    plot_sample_images()

    print(f"\nAll EDA plots saved to: {OUTPUT_DIR}")
    print("\nKey things to check:")
    print("  - Are all 3 classes well represented across splits?")
    print("  - Is one class heavily imbalanced vs others?")
    print("  - Do bbox centers cluster oddly? (indicates dataset bias)")
    print("  - Do bounding boxes look correct on sample images?")

STEP 3: EXPLORATORY DATA ANALYSIS (on clean data)
Saved: 01_split_distribution.png
Saved: 02_class_distribution.png
Saved: 03_annotation_distribution.png
Saved: 04_image_dimensions.png
Saved: 05_bbox_stats.png
Saved: 06_sample_images.png

All EDA plots saved to: ..\outputs\eda

Key things to check:
  - Are all 3 classes well represented across splits?
  - Is one class heavily imbalanced vs others?
  - Do bbox centers cluster oddly? (indicates dataset bias)
  - Do bounding boxes look correct on sample images?


## Model Training

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml
import torch

CONFIG_PATH = Path("../config/pothole.yaml")
OUTPUT_DIR = Path("../outputs/runs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def print_training_config(cfg: dict):
    print("\n--- Training Configuration ---")
    for k, v in cfg.items():
        print(f"  {k}: {v}")
    print()


def get_device():
    if torch.cuda.is_available():
        print(f"GPU detected: {torch.cuda.get_device_name(0)}")
        return 0
    else:
        print("No GPU detected. Training on CPU (slower but fine for demo).")
        return "cpu"


def train():
    device = get_device()

    # Training hyperparameters
    # For a demo project, these defaults work well.
    # Tune epochs and batch if you have more/less data.
    config = {
        "model"        : "yolov8n.pt",  # nano = fastest model
        "data"         : str(CONFIG_PATH),
        "epochs"       : 20,            # reduced from 50
        "imgsz"        : 640,
        "batch"        : 8,
        "lr0"          : 0.01,
        "lrf"          : 0.01,
        "momentum"     : 0.937,
        "weight_decay" : 0.0005,
        "warmup_epochs": 3,
        "patience"     : 5,             # reduced from 15 — stops earlier if no improvement
        "augment"      : True,
        "project"      : str(OUTPUT_DIR),
        "name"         : "pothole_v1",
        "exist_ok"     : True,
        "device"       : device,
        "verbose"      : True,
        "save"         : True,
        "save_period"  : 5,             # reduced from 10
}

    print_training_config(config)

    # Load pretrained YOLOv8 (transfer learning from COCO weights)
    model = YOLO(config.pop("model"))

    print("Starting training...\n")
    results = model.train(**config)

    # Summary after training
    print("\n" + "=" * 50)
    print("TRAINING COMPLETE")
    print("=" * 50)
    print(f"Best model : {OUTPUT_DIR}/pothole_v1/weights/best.pt")
    print(f"Last model : {OUTPUT_DIR}/pothole_v1/weights/last.pt")
    print(f"Results    : {OUTPUT_DIR}/pothole_v1/results.csv")
    print("\nOpen outputs/runs/pothole_v1/ to see:")
    print("  - results.png   (loss + mAP curves)")
    print("  - confusion_matrix.png")
    print("  - val_batch*.jpg (validation predictions)")


if __name__ == "__main__":
    print("=" * 50)
    print("STEP 4: MODEL TRAINING")
    print("=" * 50)
    train()

STEP 4: MODEL TRAINING
No GPU detected. Training on CPU (slower but fine for demo).

--- Training Configuration ---
  model: yolov8n.pt
  data: ..\config\pothole.yaml
  epochs: 20
  imgsz: 640
  batch: 8
  lr0: 0.01
  lrf: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3
  patience: 5
  augment: True
  project: ..\outputs\runs
  name: pothole_v1
  exist_ok: True
  device: cpu
  verbose: True
  save: True
  save_period: 5

Starting training...

New https://pypi.org/project/ultralytics/8.4.30 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.28  Python-3.13.5 torch-2.11.0+cpu CPU (Intel Core Ultra 7 155H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\config\pothole.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=

In [1]:
import shutil
from pathlib import Path

# source — where YOLO actually saved it
src = Path("../model/runs/outputs/runs/pothole_v1")

# destination — where it should have been saved
dst = Path("../outputs/runs/pothole_v1")

# copy entire folder
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst)

print(f"Copied from: {src}")
print(f"Copied to  : {dst}")
print("\nVerifying...")
for f in dst.rglob("*.pt"):
    print(f"Found model: {f}")

Copied from: ..\model\runs\outputs\runs\pothole_v1
Copied to  : ..\outputs\runs\pothole_v1

Verifying...
Found model: ..\outputs\runs\pothole_v1\weights\best.pt
Found model: ..\outputs\runs\pothole_v1\weights\epoch0.pt
Found model: ..\outputs\runs\pothole_v1\weights\epoch10.pt
Found model: ..\outputs\runs\pothole_v1\weights\epoch15.pt
Found model: ..\outputs\runs\pothole_v1\weights\epoch5.pt
Found model: ..\outputs\runs\pothole_v1\weights\last.pt


In [3]:
import shutil
from pathlib import Path

wrong_path = Path("../model/runs")

if wrong_path.exists():
    shutil.rmtree(wrong_path)
    print(f"Deleted: {wrong_path}")
else:
    print("Already deleted or path not found")
  

Deleted: ..\model\runs


## Evaluate

In [4]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

MODEL_PATH = Path("../outputs/runs/pothole_v1/weights/best.pt")
CONFIG_PATH = Path("../config/pothole.yaml")
OUTPUT_DIR = Path("../outputs/eda")


def evaluate_on_val():
    print("--- Validation Set Metrics ---")
    model = YOLO(str(MODEL_PATH))
    metrics = model.val(data=str(CONFIG_PATH), split="val", verbose=True)

    print(f"\n  mAP50         : {metrics.box.map50:.4f}")
    print(f"  mAP50-95      : {metrics.box.map:.4f}")
    print(f"  Precision     : {metrics.box.mp:.4f}")
    print(f"  Recall        : {metrics.box.mr:.4f}")
    return metrics


def evaluate_on_test():
    print("\n--- Test Set Metrics ---")
    model = YOLO(str(MODEL_PATH))
    metrics = model.val(data=str(CONFIG_PATH), split="test", verbose=True)

    print(f"\n  mAP50         : {metrics.box.map50:.4f}")
    print(f"  mAP50-95      : {metrics.box.map:.4f}")
    print(f"  Precision     : {metrics.box.mp:.4f}")
    print(f"  Recall        : {metrics.box.mr:.4f}")
    return metrics


def plot_training_curves():
    """Reads results.csv generated by YOLO and plots loss + mAP curves."""
    results_csv = Path("../outputs/runs/pothole_v1/results.csv")
    if not results_csv.exists():
        print("results.csv not found. Run training first.")
        return

    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()   # remove any whitespace in column names

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    axes[0][0].plot(df["epoch"], df["train/box_loss"], label="Train box loss", color="#E74C3C")
    axes[0][0].plot(df["epoch"], df["val/box_loss"], label="Val box loss", color="#3498DB")
    axes[0][0].set_title("Box loss")
    axes[0][0].legend()
    axes[0][0].set_xlabel("Epoch")

    axes[0][1].plot(df["epoch"], df["train/cls_loss"], label="Train cls loss", color="#E74C3C")
    axes[0][1].plot(df["epoch"], df["val/cls_loss"], label="Val cls loss", color="#3498DB")
    axes[0][1].set_title("Classification loss")
    axes[0][1].legend()
    axes[0][1].set_xlabel("Epoch")

    axes[1][0].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5", color="#2ECC71")
    axes[1][0].set_title("mAP@0.5 over epochs")
    axes[1][0].set_xlabel("Epoch")
    axes[1][0].set_ylim(0, 1)
    axes[1][0].legend()

    axes[1][1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision", color="#9B59B6")
    axes[1][1].plot(df["epoch"], df["metrics/recall(B)"], label="Recall", color="#F39C12")
    axes[1][1].set_title("Precision & Recall")
    axes[1][1].set_xlabel("Epoch")
    axes[1][1].set_ylim(0, 1)
    axes[1][1].legend()

    plt.suptitle("Training Metrics", fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "06_training_curves.png", dpi=150)
    plt.close()
    print(f"\nTraining curves saved to outputs/eda/06_training_curves.png")


def print_interpretation_guide():
    print("\n--- How to interpret your metrics ---")
    print("  mAP50 > 0.7   → Good model, ready for demo")
    print("  mAP50 > 0.5   → Acceptable, may miss some potholes")
    print("  mAP50 < 0.5   → Train more epochs or get more data")
    print("  High Precision + Low Recall → Model is conservative (misses potholes)")
    print("  Low Precision + High Recall → Model detects too many false potholes")
    print("  For a safety system, prefer HIGH RECALL (don't miss real potholes)")


if __name__ == "__main__":
    print("=" * 50)
    print("STEP 5: EVALUATION")
    print("=" * 50)

    evaluate_on_val()
    evaluate_on_test()
    plot_training_curves()
    print_interpretation_guide()

STEP 5: EVALUATION
--- Validation Set Metrics ---
Ultralytics 8.4.28  Python-3.13.5 torch-2.11.0+cpu CPU (Intel Core Ultra 7 155H)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 79.546.7 MB/s, size: 70.0 KB)
val: Scanning C:\Users\ASUS\Desktop\pothole\dataset_clean\valid\labels.cache... 542 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 542/542 42.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 1.1s/it 36.2s1.0ss
                   all        542       1853      0.307      0.346      0.299      0.142
       crocodile crack         63        125      0.222      0.144      0.112     0.0389
    longitudinal crack        107        284      0.238      0.222      0.165     0.0634
               pothole        533       1444      0.462      0.671      0.621      0.324
Speed: 0.6ms preprocess, 55.9ms inference, 0.0ms loss, 1.0ms po

## Prediction

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2

MODEL_PATH = Path("../outputs/runs/pothole_v1/weights/best.pt")
CONFIDENCE_THRESHOLD = 0.5   # only show detections above 50% confidence


def predict_on_image(image_path: str):
    """Run detection on a single image and save the result."""
    model = YOLO(str(MODEL_PATH))
    results = model.predict(
        source=image_path,
        conf=CONFIDENCE_THRESHOLD,
        save=True,
        project="../outputs",
        name="predictions",
        exist_ok=True,
    )
    print(f"Prediction saved to outputs/predictions/")
    return results


def predict_on_video(video_path: str):
    """Run detection on a video file and save the annotated video."""
    model = YOLO(str(MODEL_PATH))
    results = model.predict(
        source=video_path,
        conf=CONFIDENCE_THRESHOLD,
        save=True,
        project="../outputs",
        name="predictions",
        exist_ok=True,
    )
    print(f"Annotated video saved to outputs/predictions/")


def predict_live_webcam():
    model = YOLO(str(MODEL_PATH))
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("ERROR: Could not open webcam.")
        return

    print("Live webcam detection started. Press Q to quit.")

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            results = model.predict(source=frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
            annotated = results[0].plot()

            num_detections = len(results[0].boxes)
            color = (0, 0, 255) if num_detections > 0 else (0, 255, 0)
            label = f"Potholes detected: {num_detections}"
            cv2.putText(annotated, label, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2)

            cv2.imshow("Pothole Detection", annotated)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Camera released.")


if __name__ == "__main__":
    print("=" * 50)
    print("STEP 6: PREDICTION")
    print("=" * 50)

    # --- Choose one of these three modes ---

    # Mode 1: single image
    # predict_on_image("path/to/your/test_image.jpg")

    # Mode 2: video file
    # predict_on_video("path/to/road_video.mp4")

    # Mode 3: live webcam
    predict_live_webcam()

In [1]:
import cv2
cv2.destroyAllWindows()

In [2]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = Path("../outputs/runs/pothole_v1/weights/best.pt")
model = YOLO(str(MODEL_PATH))

results = model.predict(
    source="../dataset/test/images",
    conf=0.25,
    save=True,
    project="../outputs",
    name="predictions",
    exist_ok=True
)
print(f"Done! Check outputs/predictions/")


image 1/274 c:\Users\ASUS\Desktop\pothole\model\..\dataset\test\images\img-104_jpg.rf.7f5a44e182a85aa8e94a769c8160d443.jpg: 640x640 1 pothole, 101.6ms
image 2/274 c:\Users\ASUS\Desktop\pothole\model\..\dataset\test\images\img-111_jpg.rf.9202c3a0b242ca295877ca35b866a1c7.jpg: 640x640 2 potholes, 67.4ms
image 3/274 c:\Users\ASUS\Desktop\pothole\model\..\dataset\test\images\img-124_jpg.rf.dd9ac0bd462c0324657c628d74aba86d.jpg: 640x640 4 potholes, 153.9ms
image 4/274 c:\Users\ASUS\Desktop\pothole\model\..\dataset\test\images\img-125_jpg.rf.9e4f594e1742069371d3d65fdb9d61df.jpg: 640x640 1 pothole, 224.1ms
image 5/274 c:\Users\ASUS\Desktop\pothole\model\..\dataset\test\images\img-130_jpg.rf.621b9f58b59928c3fbadec65559dcc30.jpg: 640x640 2 potholes, 186.8ms
image 6/274 c:\Users\ASUS\Desktop\pothole\model\..\dataset\test\images\img-137_jpg.rf.13b2055e9b032265a1c4b7f5b85509c2.jpg: 640x640 2 potholes, 305.7ms
image 7/274 c:\Users\ASUS\Desktop\pothole\model\..\dataset\test\images\img-138_jpg.rf.1175